# Paper 2 — Task 6: Filler Recovery and Verification (Stages 6a + 6b)

**Runtime: Phase 1 is CPU, minutes. Phase 2 is API, ~$1 per model.**

This is the rebuilt version of `Untitled10.ipynb`. That notebook had grown to
30 cells and ~5,400 lines covering the entire investigation — the live Task 6
code, three superseded versions of the Task 1 probe, the retired scope
tooling, an invalid horizon estimator, and a dozen one-off diagnostics. Most
of it had already been extracted into standalone scripts; what remained was
hard to run without knowing which cells were current.

**This notebook does one job: build pile B.** Nothing was deleted — see
`Untitled10_REBUILD_MANIFEST.md` for where all 30 original cells went.

---

## What Task 6 is for

Revision ground truth in this pipeline is a **surface marker word** — "wait",
"actually", "alternatively". So a probe that separates revision windows from
arbitrary control windows might only be detecting *the word*, not the
revision. That is the lexical confound.

The fix is a matched contrast. Two piles, identical in every respect except
GPT-4o's verdict:

| | marker word | position | model | extraction | revision? |
|---|---|---|---|---|---|
| **Pile A** (Stage 3) | "wait" | at the marker | same | same | yes |
| **Pile B** (here) | "wait" | at the marker | same | same | **no** |

Holding the word constant is what makes it a control. The gap between the
confounded contrast and the matched contrast is exactly the portion of the
original signal that was lexical anticipation.

## The two phases here

- **Phase 1 (CPU)** replays Task 2's *exact* detection lexicon to recover the
  candidates GPT-4o rejected → `rejects/{model}_rejects.json`
- **Phase 2 (API)** has GPT-4o confirm those rejects really are non-revisions,
  so pile B is verified rather than merely unverified →
  `rejects/{model}_confirmed_filler.json`

Phase 2 matters more than it looks. Without it, pile B is "things one pass
didn't flag", which is not the same as "confirmed non-revisions" — and the
whole control rests on pile B being genuinely revision-free.

## What happens after this

Phases 3 and 4 used to live in this notebook as commented-out scaffolding.
They are now proper notebooks:

- **Phase 3** → `Paper2_1_Filler_Activation_Extraction.ipynb` (GPU, per model)
- **Phase 4** → `Paper2_2_Construct_Validity_Probe.ipynb` (CPU, Table 1)

**Prerequisites:** Stage 1 traces and Stage 2 labels must already exist.
Phase 2 needs `OPENAI_API_KEY` set in Colab's secrets (the key icon in the
left sidebar).


In [ ]:
# ---- Google Drive + paths ---------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")

## Setup — imports, models, paths


In [ ]:
import os, re, json, gc, time, random
import numpy as np
from pathlib import Path
from collections import defaultdict

from google.colab import userdata
GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
TRACE_BASE  = GDRIVE_BASE / "traces"
LABEL_BASE  = GDRIVE_BASE / "labels"
ACTIV_BASE  = GDRIVE_BASE / "activations"
REJECT_BASE = GDRIVE_BASE / "rejects"          # NEW: pile B
RESULTS_DIR = GDRIVE_BASE / "results_v2"
REJECT_BASE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    "DeepSeek-R1-Distill-Llama-8B",
    "DeepSeek-R1-Distill-Qwen-1.5B",
    "DeepSeek-R1-Distill-Qwen-7B",
    "DeepSeek-R1-Distill-Qwen-14B",
]

WINDOW_TOKENS = 30
WINDOW_CENTER = 30
SEED = 42

---

# Phase 1 (CPU) — recover the rejects

The lexicon below **must match Task 2's exactly.** If it drifts, pile B is no longer the complement of pile A and the control is void.

Note `alternatively[,.]` is present here. An earlier suppression ban list was built from a *different*, earlier regex that omitted it, which caused a real leak — this is the authoritative list, with `type_hint` hardcoded per pattern.


In [ ]:
# EXACT lexicon from Task 2 -- must match or the candidate set will differ
REVISION_PATTERNS = [
    (r"let me (?:start|try|approach|think about) (?:this )?(?:again|differently|from scratch|over)", "global"),
    (r"(?:i need to|let me) (?:completely |totally )?reconsider", "global"),
    (r"(?:actually|wait)[,.]?\s+(?:i think )?(?:my approach|this approach|my method) (?:is wrong|doesn'?t work|is incorrect)", "global"),
    (r"(?:let me|i'?ll) (?:try a |use a )?(?:different|new|another) (?:approach|method|strategy|way)", "global"),
    (r"i (?:made|think i made) (?:a )?(?:mistake|an error|a wrong assumption)", "global"),
    (r"this (?:doesn'?t|won'?t) work", "global"),
    (r"let me (?:restart|start over|begin again)", "global"),
    (r"on second thought", "global"),
    (r"wait,?\s+(?:actually|let me|no)", "local"),
    (r"actually,?\s+(?:wait|no|let me|that'?s)", "local"),
    (r"(?:hmm+|hm)[,.]?\s+(?:actually|wait|let me|so)", "local"),
    (r"let me (?:re-?check|re-?examine|re-?verify|double.?check)", "local"),
    (r"that'?s (?:not right|wrong|incorrect|not correct)", "local"),
    (r"i was wrong", "local"),
    (r"let me recalculate", "local"),
    (r"(?:wait|actually)[,.]?\s+i (?:think i )?(?:made an error|got that wrong|miscalculated)", "local"),
    (r"\bwait\b(?!\s+for)", "local"),
    (r"\bactually\b", "local"),
    (r"let me (?:re-?think|think (?:about )?this (?:more carefully|again))", "local"),
    (r"alternatively[,.]", "local"),
    (r"no[,.]?\s+wait", "local"),
]
COMPILED = [(re.compile(p, re.IGNORECASE), h) for p, h in REVISION_PATTERNS]
MAX_CANDIDATES_PER_TRACE = 50   # must match Task 2

### Phase 1 helpers


In [ ]:
def clean_text(t):
    return t.replace("Ġ", " ").replace("Ċ", "\n").replace("ĉ", "\t")


def get_analysis_text(trace):
    think = clean_text(trace.get("think_content", ""))
    clean = clean_text(trace.get("response_clean", ""))
    return think if think.strip() else clean


def stage1_candidates(text, max_candidates=MAX_CANDIDATES_PER_TRACE):
    """Re-run Stage 1 EXACTLY as Task 2 did. Deterministic."""
    cands, seen = [], set()
    for pat, hint in COMPILED:
        for m in pat.finditer(text):
            s = m.start()
            if any(abs(s - x) < 20 for x in seen):
                continue
            seen.add(s)
            cands.append({
                "char_start":   s,
                "char_end":     m.end(),
                "matched_text": m.group(),
                "type_hint":    hint,
            })
    cands.sort(key=lambda c: c["char_start"])
    return cands[:max_candidates]


def normalize_marker(txt):
    """Collapse a matched span to its head marker word, for stratification."""
    t = txt.lower().strip()
    for w in ("wait", "actually", "hmm", "alternatively", "no"):
        if t.startswith(w):
            return w
    return t.split()[0] if t.split() else "other"


def char_to_word_index(text, char_pos):
    return len(text[:char_pos].split())


def recover_rejects(model_short):
    """
    Rejects = Stage-1 candidates MINUS the accepted revisions in the label JSON.
    Matched on char_start (exact -- both come from the same deterministic regex).
    """
    trace_dir = TRACE_BASE / model_short
    label_dir = LABEL_BASE / model_short
    out = []

    label_files = {
        f.stem: f for f in label_dir.glob("*.json")
        if f.name not in ("labeling_log.json", "label_summary.json")
    }

    for pid, lf in sorted(label_files.items()):
        with open(lf) as f:
            label = json.load(f)
        if label.get("skipped"):
            continue

        tf = trace_dir / f"{pid}.json"
        if not tf.exists():
            continue
        with open(tf) as f:
            trace = json.load(f)

        text  = get_analysis_text(trace)
        cands = stage1_candidates(text)

        accepted_starts = {r["char_start"] for r in label.get("revisions", [])}

        for c in cands:
            if c["char_start"] in accepted_starts:
                continue                      # this one was ACCEPTED -> pile A
            # -> pile B: same marker word, judged NOT a revision
            w = char_to_word_index(text, c["char_start"])
            out.append({
                "problem_id":   pid,
                "difficulty":   label.get("difficulty", "unknown"),
                "char_start":   c["char_start"],
                "matched_text": c["matched_text"],
                "marker":       normalize_marker(c["matched_text"]),
                "word_idx":     max(0, w - 1),   # token BEFORE marker -- same as pile A
                "type_hint":    c["type_hint"],
            })

    return out

### Run Phase 1

Check the printed counts: you want ≥ ~200 rejects per model for a usable probe, and ≥ ~100 of a single marker word to stratify on.


In [ ]:
print("PHASE 1 -- recovering rejects (pile B)\n" + "=" * 60)
reject_counts = {}
for m in MODELS:
    rej = recover_rejects(m)
    reject_counts[m] = len(rej)

    # group by marker so we can see if there are enough per word to stratify
    by_marker = defaultdict(int)
    for r in rej:
        by_marker[r["marker"]] += 1

    with open(REJECT_BASE / f"{m}_rejects.json", "w") as f:
        json.dump(rej, f, indent=2)

    print(f"\n{m}")
    print(f"  rejects recovered: {len(rej)}")
    for mk, c in sorted(by_marker.items(), key=lambda x: -x[1])[:8]:
        print(f"    {mk:15s}: {c}")

print("\n  Saved to:", REJECT_BASE)
print("\n  >>> Check counts. You need >= ~200 rejects per model for a")
print("      usable probe, and >= ~100 of a single marker word to stratify.")

---

# Phase 2 (API) — confirm the rejects are non-revisions

Needs `OPENAI_API_KEY` in Colab secrets. `MAX_VERIFY_PER_MODEL` is a cost cap at roughly \$1 per model — raise it deliberately, not by accident.


In [ ]:
RUN_PHASE_2 = True          # set False to skip and use all rejects unfiltered
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")#os.environ.get("OPENAI_API_KEY", "PASTE_KEY_HERE")
GPT_MODEL      = "gpt-4o"
MAX_VERIFY_PER_MODEL = 900  # cost cap; ~$1 per model

VERIFY_PROMPT = """You are analyzing an excerpt from a language model's reasoning trace.

At the MARKED position, the model uses a word like "wait", "actually", or "hmm".
Determine whether this marks a GENUINE PLAN REVISION or is merely DISCOURSE FILLER.

GENUINE REVISION: the model realizes its approach or calculation is wrong and
changes course -- abandons a line of reasoning, corrects a fundamental assumption.

DISCOURSE FILLER: the word appears but the model continues the SAME reasoning.
Examples: "wait" used for emphasis, "actually" as a hinge, "hmm" as thinking-aloud,
a brief restatement that does not change the approach.

Respond ONLY with JSON:
{"verdict": "REVISION" | "FILLER", "confidence": 0.0-1.0}"""

### Run Phase 2


In [ ]:
def verify_rejects(model_short, rejects, cap=MAX_VERIFY_PER_MODEL):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    trace_dir  = TRACE_BASE / model_short
    text_cache = {}
    rng = random.Random(SEED)

    sample    = rejects if len(rejects) <= cap else rng.sample(rejects, cap)
    confirmed = []
    n_failed  = 0                      # <-- track failures

    for i, r in enumerate(sample):
        pid = r["problem_id"]
        if pid not in text_cache:
            with open(trace_dir / f"{pid}.json") as f:
                text_cache[pid] = get_analysis_text(json.load(f))
        text = text_cache[pid]

        cs     = r["char_start"]
        before = " ".join(text[:cs].split()[-80:])
        after  = " ".join(text[cs:].split()[:80])
        user   = (f"--- BEFORE ---\n{before}\n\n"
                  f"*** MARKED: '{r['matched_text']}' ***\n\n"
                  f"--- AFTER ---\n{after}\n\nRespond with the JSON.")

        ok = False
        for attempt in range(3):
            try:
                resp = client.chat.completions.create(
                    model=GPT_MODEL,
                    messages=[{"role": "system", "content": VERIFY_PROMPT},
                              {"role": "user",   "content": user}],
                    max_tokens=60, temperature=0.0,
                    response_format={"type": "json_object"},
                )
                res = json.loads(resp.choices[0].message.content)
                if res.get("verdict") == "FILLER" and res.get("confidence", 0) >= 0.8:
                    confirmed.append({**r, "filler_confidence": res["confidence"]})
                ok = True
                break
            except Exception as e:
                if attempt == 2:                       # <-- surface it
                    print(f"    [FAIL {i}] {type(e).__name__}: {e}")
                time.sleep(5 * (2 ** attempt))

        if not ok:
            n_failed += 1
            if n_failed >= 10:                          # <-- abort, don't limp on
                raise RuntimeError(
                    f"ABORTING: {n_failed} consecutive API failures. "
                    f"Confirmed only {len(confirmed)} so far. Check quota/key."
                )

        time.sleep(0.5)
        if (i + 1) % 50 == 0:
            print(f"    verified {i+1}/{len(sample)} -> "
                  f"{len(confirmed)} confirmed, {n_failed} failed")

    print(f"    DONE: {len(confirmed)} confirmed, {n_failed} failed "
          f"out of {len(sample)}")
    return confirmed

if RUN_PHASE_2:
    print("\n\nPHASE 2 -- GPT-4o confirms rejects are genuine non-revisions")
    print("=" * 60)
    for m in MODELS:
        with open(REJECT_BASE / f"{m}_rejects.json") as f:
            rej = json.load(f)
        print(f"\n{m}  ({len(rej)} rejects, verifying up to {MAX_VERIFY_PER_MODEL})")
        conf = verify_rejects(m, rej)
        with open(REJECT_BASE / f"{m}_confirmed_filler.json", "w") as f:
            json.dump(conf, f, indent=2)
        print(f"  -> {len(conf)} confirmed FILLER (these are the control set)")

---

## Done — next steps

You should now have `rejects/{model}_confirmed_filler.json` for all four models.

1. **`Paper2_1_Filler_Activation_Extraction.ipynb`** — Phase 3, GPU, once per model. Extracts pile B's activation windows.
2. **`Paper2_2_Construct_Validity_Probe.ipynb`** — Phase 4, CPU. The matched probe, and Table 1.

Expect `activations_filler/` to hold **fewer** events than the confirmed-filler JSON lists: Phase 3 drops any event whose window runs short at a sequence boundary rather than zero-padding it. Do not assume 1:1.
